# Thanh điệu tiếng Việt → giai điệu: lấy dữ liệu từ bài hát thật

Notebook này dựng bảng dữ liệu **mỗi chữ một dòng** (chữ, thanh điệu, cao độ nốt hát, nhịp) từ bản thu bài hát,
rồi học bảng xác suất *"chữ thanh này đi sau chữ thanh kia thì giai điệu thường đi lên hay xuống, bao xa"*.
App dùng bảng này để viết giai điệu theo lời và cảnh báo chỗ hát lên dễ nghe thành chữ khác.

Các bước cho mỗi bài:

1. **Tách giọng hát** khỏi nhạc nền bằng Demucs (`htdemucs`).
2. **Dò cao độ (F0)** của giọng hát bằng torchcrepe (cần GPU) hoặc pYIN.
3. **Chia thành nốt**: cắt ở chỗ ngừng hát hoặc chỗ cao độ đổi hẳn.
4. **Căn lời**: mô hình nhận dạng tiếng Việt wav2vec2 (CTC) cho biết mỗi chữ được hát từ giây nào tới giây nào.
5. **Ghép chữ với nốt**: chữ lấy các nốt nằm trong khoảng thời gian của nó (nhiều nốt = chữ được ngân luyến).

Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy. Dữ liệu chỉ dùng cho học tập cá nhân;
audio và lời để trên Google Drive, **không đưa lên git**.

## 1. Chuẩn bị: GPU, Google Drive và mã nguồn

In [ ]:
!nvidia-smi -L || echo "Không có GPU: vẫn chạy được nhưng chậm (dùng pYIN thay torchcrepe)"
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/aicomposer_vn_tone'  # đổi nếu muốn để chỗ khác
import os
for sub in ['audio', 'kar', 'processed_audio', 'processed_kar']:
    os.makedirs(f'{DATA}/{sub}', exist_ok=True)

In [ ]:
REPO_URL = 'https://github.com/<tai-khoan-cua-ban>/aicomposer.git'  # sửa thành repo của bạn
!git clone -q $REPO_URL /content/aicomposer || (cd /content/aicomposer && git pull -q)
%cd /content/aicomposer

In [ ]:
# Thư viện nhẹ (giống laptop) + thư viện nặng chỉ cần ở Colab
%pip install -q -r ml/requirements.txt
%pip install -q demucs torchcrepe "transformers>=4.40" yt-dlp

## 2. Đặt bài hát vào Drive

Cấu trúc thư mục (mỗi tác giả/ca sĩ một thư mục con, để lúc đánh giá tách train/test theo người):

```
aicomposer_vn_tone/audio/
    trinh_cong_son/
        diem_xua.mp3
        diem_xua.lrc      ← lời có mốc thời gian từng câu (tốt nhất), hoặc diem_xua.txt (mỗi dòng một câu)
    van_cao/
        ...
```

File `.lrc` lấy từ các trang lời bài hát hoặc tự gõ. Không có mốc thời gian thì dùng `.txt`, máy căn cả bài một lần (chậm hơn, dễ trượt hơn).

Muốn tải bằng yt-dlp: tạo `audio/urls.csv` với các cột `url,song_id,artist_id` rồi chạy ô dưới (lời vẫn phải tự thêm,
đặt tên `<song_id>.lrc` cạnh file audio).

In [ ]:
import os
urls = f'{DATA}/audio/urls.csv'
if os.path.exists(urls):
    !python ml/vn_tone/extract_audio.py --urls "$urls" --audio-dir "$DATA/audio" --download-only
else:
    print('Không có urls.csv: bỏ qua bước tải.')
!find "$DATA/audio" -type f | head -20

## 3. Chạy thử từng bước trên một bài

Phần này để hiểu từng bước làm gì và kiểm tra kết quả bằng tai, bằng mắt trước khi chạy cả thư mục.

In [ ]:
import sys, glob
from pathlib import Path
sys.path.insert(0, 'ml/vn_tone')
import extract_audio as ea
from IPython.display import Audio, display

pairs = [(Path(a), Path(a).with_suffix(ext)) for a in sorted(glob.glob(f'{DATA}/audio/**/*.*', recursive=True))
         if Path(a).suffix.lower() in ea.AUDIO_EXT for ext in ('.lrc', '.txt') if Path(a).with_suffix(ext).exists()]
audio_path, lyrics_path = pairs[0]
print('Bài thử:', audio_path.name, '| lời:', lyrics_path.name)

# Bước 1: tách giọng (lần đầu tải mô hình Demucs, mất vài phút)
vocals_path = ea.separate_vocals(audio_path, Path('/content/work/demucs'))
vocals = ea.load_wav(vocals_path)
display(Audio(vocals[: ea.SR * 20], rate=ea.SR))  # nghe 20 giây đầu: chỉ còn giọng hát

In [ ]:
# Bước 2 + 3: cao độ và nốt
import matplotlib.pyplot as plt
times, midi, conf = ea.track_f0(vocals, ea.SR, 'auto')
notes = ea.notes_from_f0(times, midi)
print(f'{len(notes)} nốt')
seg = times < 30
plt.figure(figsize=(14, 4))
plt.plot(times[seg], midi[seg], lw=0.8, label='F0 (MIDI)')
for on, off, p in notes:
    if on < 30:
        plt.hlines(p, on, off, colors='r', lw=3)
plt.xlabel('giây'); plt.ylabel('cao độ MIDI'); plt.title('30 giây đầu: đường cao độ và nốt'); plt.legend(); plt.show()

In [ ]:
# Bước 4: căn lời bằng wav2vec2 tiếng Việt (lần đầu tải mô hình ~400 MB)
aligner = ea.Aligner(ea.DEFAULT_ASR)
lines = ea.parse_lyrics(lyrics_path.read_text(encoding='utf-8'))
t0, words = lines[0]
if t0 is not None:
    t1 = lines[1][0] if len(lines) > 1 else t0 + 10
    a = max(0.0, t0 - 0.3)
    aligned = aligner.align_words(vocals[int(a * ea.SR): int(t1 * ea.SR)], words, offset=a)
else:
    aligned = aligner.align_words(vocals[: ea.SR * 30], words)
for _, w, s, e in aligned:
    print(f'{w:10s} {s:7.2f}s → {e:7.2f}s')

In [ ]:
# Bước 5: ghép chữ với nốt cho câu đầu
import pandas as pd
from vn_common import finish_rows
bpm = ea.estimate_bpm(ea.load_wav(audio_path))
sylls = ea.match_words_to_notes([[(w, s, e) for _, w, s, e in aligned]], notes, bpm)
pd.DataFrame(finish_rows(sylls, 'thu', 'thu', 'audio', (0, 'major')))

## 4. Chạy cả thư mục

Mỗi bài mất khoảng 1–2 phút trên T4. Kết quả: `processed_audio/syllables.parquet` và `manifest.csv`
(bài nào lỗi, vì sao).

In [ ]:
!python ml/vn_tone/extract_audio.py --audio-dir "$DATA/audio" --out "$DATA/processed_audio" --work /content/work
import pandas as pd
pd.read_csv(f'{DATA}/processed_audio/manifest.csv')

## 5. (Tuỳ chọn) Thêm file karaoke .kar/.mid hoặc MusicXML

File karaoke đã có sẵn nốt và lời nên chính xác hơn audio. Đặt vào `aicomposer_vn_tone/kar/<tac_gia>/...`.
Script tự đoán bảng mã của lời (UTF-8, Windows-1258, VNI).

In [ ]:
!python ml/vn_tone/extract_symbolic.py --input "$DATA/kar" --out "$DATA/processed_kar" || echo "Chưa có file karaoke"

## 6. Huấn luyện và đánh giá

`train.py` chia train/test **theo tác giả** (người viết trong tập test chưa xuất hiện khi học), in log loss và độ chính xác
hướng giai điệu so với luật cũ của app (TONE_HEIGHT), rồi học lại trên toàn bộ dữ liệu và ghi `tone_model.json`.
Cài thêm `lightgbm` nếu muốn có mô hình phụ để so sánh trong báo cáo.

In [ ]:
%pip install -q lightgbm
import os
args = [f'--data "{DATA}/processed_audio"']
if any(f.startswith('syllables') for f in os.listdir(f'{DATA}/processed_kar')):
    args.append(f'--data "{DATA}/processed_kar"')
args = ' '.join(args)
!python ml/vn_tone/train.py $args --out "$DATA/tone_model.json"

## 7. Đưa vào app

Tải `tone_model.json` về, chép vào `frontend/public/models/tone_model.json` trong repo rồi commit. Mở app là tự nạp;
xoá file thì app quay về luật cũ.

In [ ]:
from google.colab import files
files.download(f'{DATA}/tone_model.json')